Step One: Process dataset that will be used for the final test

In [23]:
import pandas as pd
import os
import glob

# Process dataset: only keep columns 'title' and 'label (real)'
df_fakenews = pd.read_csv('FakeNewsNet.csv')
df_fakenews_clean = df_fakenews[['title', 'real']].copy()

# Rename columns to match standard format
df_fakenews_clean = df_fakenews_clean.rename(columns={
    'title': 'text', 
    'real': 'label'
})

# Since dataset originally maps 'Real' to 1, change it so that 'Real' maps to 0 to match model data format
df_fakenews_clean['label'] = df_fakenews_clean['label'].map({1: 0, 0: 1})

# Only keep 500 from each category
fake_subset = df_fakenews_clean[df_fakenews_clean['label'] == 1].head(500)
real_subset = df_fakenews_clean[df_fakenews_clean['label'] == 0].head(500)

# Combine them together
balanced_df = pd.concat([fake_subset, real_subset])

# Shuffle the data
shuffled_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Verify the counts
print(shuffled_df['label'].value_counts())

# Verify the final format
print(shuffled_df.head())

label
0    500
1    500
Name: count, dtype: int64
                                                text  label
0  Teen Mom OG's Tyler Baltierra Reveals How He L...      0
1  Emma Watson donates £1 million to anti-sexual ...      0
2  Cole Sprouse and Lili Reinhart's Relationship ...      0
3  Kate Middleton Wears an Emilia Wickstead Plaid...      0
4  Noel Gallagher slams Harry Styles' solo debut:...      1


Step Two: Clean data in 'text' column by removing stopwords, special characters, and hashtags. Covert all letters to lowercase, and enforce 50 word limit

In [24]:
import re
import torch
import nltk
from nltk.corpus import stopwords
from torch.utils.data import Dataset, DataLoader
from collections import Counter

# Download the standard stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sissi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [25]:
def clean_and_limit_text(text):
    # Convert to string, then lowercase
    text = str(text).lower()
    
    # Remove hashtags and URLs
    text = re.sub(r'#\w+', '', text) 
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Remove special characters
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Tokenize by splitting into words and remove stopwords
    words = text.split()
    cleaned_words = [word for word in words if word not in stop_words]
    
    # Enforce 50 word limit
    cleaned_words = cleaned_words[:50]
    
    return cleaned_words

shuffled_df['clean_tokens'] = shuffled_df['text'].apply(clean_and_limit_text)

In [26]:
print(shuffled_df.head())

                                                text  label  \
0  Teen Mom OG's Tyler Baltierra Reveals How He L...      0   
1  Emma Watson donates £1 million to anti-sexual ...      0   
2  Cole Sprouse and Lili Reinhart's Relationship ...      0   
3  Kate Middleton Wears an Emilia Wickstead Plaid...      0   
4  Noel Gallagher slams Harry Styles' solo debut:...      1   

                                        clean_tokens  
0  [teen, mom, ogs, tyler, baltierra, reveals, lo...  
1  [emma, watson, donates, million, antisexual, h...  
2  [cole, sprouse, lili, reinharts, relationship,...  
3  [kate, middleton, wears, emilia, wickstead, pl...  
4  [noel, gallagher, slams, harry, styles, solo, ...  


Step Three: Calculate and record statistics

In [27]:
# Calculate real/fake claim counts
class_counts = shuffled_df['label'].value_counts()

print("Cleaned Data Statistics: ")
print(f"Total samples: {len(shuffled_df)}")
print(f"Real Claims (0): {class_counts.get(0, 0)}")
print(f"Fake Claims (1): {class_counts.get(1, 0)}")

print("\nExample Cleaned Sample: ")
print(f"Original: {shuffled_df.iloc[0]['text']}")
print(f"Cleaned & Tokenized: {shuffled_df.iloc[0]['clean_tokens']}")
print(f"Label: {shuffled_df.iloc[0]['label']}")

Cleaned Data Statistics: 
Total samples: 1000
Real Claims (0): 500
Fake Claims (1): 500

Example Cleaned Sample: 
Original: Teen Mom OG's Tyler Baltierra Reveals How He Lost 30 lbs.
Cleaned & Tokenized: ['teen', 'mom', 'ogs', 'tyler', 'baltierra', 'reveals', 'lost', 'lbs']
Label: 0


Step Four: Convert to numerical tensors

In [28]:
# Create a vocabulary for all tokens in the dataset
all_words = [word for tokens in shuffled_df['clean_tokens'] for word in tokens]
vocab_counts = Counter(all_words)

# Map each word to a integer. Start at 1 to save 0 for padding 
vocab_to_int = {word: i+1 for i, (word, count) in enumerate(vocab_counts.items())}

def text_to_ints(tokens):
    return [vocab_to_int[word] for word in tokens]

shuffled_df['numerical_tokens'] = shuffled_df['clean_tokens'].apply(text_to_ints)

Step Five: Pad the sequences

In [29]:
SEQ_LENGTH = 50

def pad_features(numerical_tokens, seq_length):
    # Create an array of zeros and get length of token list
    features = torch.zeros(seq_length, dtype=torch.int64)
    token_len = len(numerical_tokens)
    
    # If the sentence is empty after cleaning, return zeros
    if token_len == 0:
        return features
        
    # Place the tokens into the tensor
    features[:token_len] = torch.tensor(numerical_tokens)
    return features

# Apply padding
padded_tensors = torch.stack(
    shuffled_df['numerical_tokens'].apply(lambda x: pad_features(x, SEQ_LENGTH)).tolist()
)

# Extract labels as a tensor
labels_tensor = torch.tensor(shuffled_df['label'].values, dtype=torch.float32)

Step Six: Wrap in Pytorch Dataset and DataLoader

In [30]:
class MedicalMisinfoDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

full_dataset = MedicalMisinfoDataset(padded_tensors, labels_tensor)

# Wrap it in a DataLoader for batching (32 samples per batch)
BATCH_SIZE = 32
train_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Test the loader
dataiter = iter(train_loader)
sample_x, sample_y = next(dataiter)

print("\nDataLoader Verification: ")
print(f"Batch X shape: {sample_x.shape}")
print(f"Batch Y shape: {sample_y.shape}")


DataLoader Verification: 
Batch X shape: torch.Size([32, 50])
Batch Y shape: torch.Size([32])


Step Seven: Save the cleaned and processed dataset

In [32]:
# Save to dataset_csv folder without the index column (better readability)
csv_folder = 'dataset\\dataset_csv'
csv_file_path = os.path.join(csv_folder, "cleaned_claims_for_final_test.csv")
shuffled_df.to_csv(csv_file_path, index=False)
print(f"Dataframe successfully saved to: {csv_file_path}")

# Save PyTorch Tensors for later use
tensor_folder = 'dataset\\dataset_tensors'
tensor_file_path = os.path.join(tensor_folder, "final_test_tensors.pt")

torch.save({
    'features': padded_tensors,
    'labels': labels_tensor
}, tensor_file_path)
print(f"PyTorch tensors successfully saved to: {tensor_file_path}")

Dataframe successfully saved to: dataset\dataset_csv\cleaned_claims_for_final_test.csv
PyTorch tensors successfully saved to: dataset\dataset_tensors\final_test_tensors.pt
